In [2]:
# ============================================================
# FIX FAISS / OPENMP CRASH
# ============================================================

import os

os.environ["OMP_NUM_THREADS"] = "1"

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

print(
    "OpenMP Fix Applied"
)

OpenMP Fix Applied


In [3]:
# ============================================================
# IMPORT REQUIRED LIBRARIES
# ============================================================

import pandas as pd

from dotenv import load_dotenv

from langchain_openai import OpenAIEmbeddings

from langchain_community.vectorstores import FAISS

from langchain_ollama import ChatOllama

print(
    "Libraries Loaded Successfully"
)

Libraries Loaded Successfully


/var/folders/vq/9swrpcvn7sl3cb_pcdyf9r3h0000gn/T/ipykernel_1964/2542142073.py:11: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [4]:
# ============================================================
# LOAD ENVIRONMENT VARIABLES
# ============================================================

load_dotenv()

print(
    "Environment Variables Loaded"
)

Environment Variables Loaded


In [6]:
# ============================================================
# LOAD EVALUATION DATASET
# ============================================================

eval_df = pd.read_csv(
    "../notebooks/evaluation_dataset.csv"
)

print(
    "Rows:",
    len(eval_df)
)

eval_df.head()

Rows: 1000


,query,article,reference_summary
0,solar observatory aims to provide better under...,cnn nasa has postponed for one day the schedul...,solar observatory aims to provide better under...
1,mary j bliges new album my life ii,cnn seventeen years after the release of her b...,mary j bliges new album my life ii is a follow...
2,germany coach joachim low claims spain are fav...,cnn germany coach joachim low has tried to lif...,germany coach joachim low claims spain are fav...
3,presidential historians weigh in on how histor...,cnn with record low approval ratings and inten...,presidential historians weigh in on how histor...
4,chinas gold medal gymnasts cleared of competin...,cnn chinas olympic gold medal gymnasts have be...,chinas gold medal gymnasts cleared of competin...


In [7]:
# ============================================================
# VERIFY DATASET
# ============================================================

print(
    eval_df.columns.tolist()
)

['query', 'article', 'reference_summary']


In [8]:
# ============================================================
# LOAD OPENAI EMBEDDINGS
# ============================================================

embeddings = OpenAIEmbeddings(

    model="text-embedding-3-small"

)

print(
    "Embeddings Loaded Successfully"
)

Embeddings Loaded Successfully


In [9]:
# ============================================================
# LOAD FAISS VECTOR STORE
# ============================================================

vector_store = FAISS.load_local(

    "../notebooks/faiss_20k",

    embeddings,

    allow_dangerous_deserialization=True

)

print(
    "FAISS Loaded Successfully"
)

FAISS Loaded Successfully


In [10]:
# ============================================================
# TEST RETRIEVAL
# ============================================================

docs = vector_store.similarity_search(

    "artificial intelligence",

    k=5

)

print(
    "Retrieved Documents:",
    len(docs)
)

Retrieved Documents: 5


In [11]:
# ============================================================
# LOAD LLAMA3
# ============================================================

llm = ChatOllama(

    model="llama3",

    temperature=0

)

print(
    "Llama3 Loaded Successfully"
)

Llama3 Loaded Successfully


In [12]:
# ============================================================
# TEST LLAMA3
# ============================================================

response = llm.invoke(
    "Say Hello"
)

print(
    response.content
)

Hello! It's nice to meet you. Is there something I can help you with, or would you like to chat?


In [13]:
# ============================================================
# RESET RESULTS
# ============================================================

results = []

print(
    "Results Reset Successfully"
)

Results Reset Successfully


In [15]:
response = llm.invoke(
    "What is Artificial Intelligence?"
)

print(
    response.content
)

Artificial intelligence (AI) refers to the development of computer systems that can perform tasks that would typically require human intelligence, such as:

1. Learning: AI systems can learn from data and improve their performance over time.
2. Reasoning: AI systems can draw conclusions based on available information and make decisions.
3. Problem-solving: AI systems can identify problems and develop solutions.

AI is a broad field that encompasses various subfields, including:

1. Machine learning (ML): A type of AI that enables machines to learn from data without being explicitly programmed.
2. Natural language processing (NLP): A type of AI that enables computers to understand, interpret, and generate human-like text or speech.
3. Computer vision: A type of AI that enables computers to interpret and understand visual information from images and videos.

AI systems can be categorized into three types based on their level of autonomy:

1. Narrow or weak AI: These systems are designed 

In [16]:
query = "artificial intelligence"

docs = vector_store.similarity_search(
    query,
    k=5
)

context = "\n\n".join(
    [doc.page_content for doc in docs]
)

print(
    len(context)
)

2480


In [17]:
import time

start = time.time()

response = llm.invoke(
    "What is Artificial Intelligence?"
)

print(
    "Seconds:",
    round(
        time.time() - start,
        2
    )
)

Seconds: 83.13


500 × 83 sec
=
41500 sec
=
691 min
=
11.5 hours

GPT4o → 100 rows
Llama3 → 25 rows
Mistral → 25 rows

In [18]:
# ============================================================
# DENSE LLAMA3 EVALUATION PIPELINE
# ============================================================

for idx, row in eval_df.iterrows():

    if idx >= 25:
        break

    if idx % 50 == 0:

        print(
            f"Processing Row {idx}"
        )

    query = str(
        row["query"]
    )

    reference_summary = str(
        row["reference_summary"]
    )

    # --------------------------------------------------------
    # DENSE RETRIEVAL
    # --------------------------------------------------------

    docs = vector_store.similarity_search(

        query,

        k=5

    )

    context = "\n\n".join(

        [

            doc.page_content

            for doc in docs

        ]

    )

    # --------------------------------------------------------
    # PROMPT
    # --------------------------------------------------------

    prompt = f"""

    You are a helpful AI assistant.

    Use ONLY the provided context.

    Context:

    {context}

    Question:

    {query}

    Answer:

    """

    # --------------------------------------------------------
    # GENERATE RESPONSE
    # --------------------------------------------------------

    response = llm.invoke(
        prompt
    )

    answer = response.content

    # --------------------------------------------------------
    # STORE RESULTS
    # --------------------------------------------------------

    results.append({

        "pipeline":
        "Dense Llama3",

        "model":
        "Llama3",

        "retrieval_method":
        "Dense",

        "query":
        query,

        "reference_summary":
        reference_summary,

        "generated_answer":
        answer

    })

print(
    "\nDense Llama3 Evaluation Completed"
)

Processing Row 0

Dense Llama3 Evaluation Completed


In [19]:
# ============================================================
# CREATE RESULTS DATAFRAME
# ============================================================

results_df = pd.DataFrame(
    results
)

print(
    "Total Records:",
    len(results_df)
)

results_df.head()

Total Records: 53


,pipeline,model,retrieval_method,query,reference_summary,generated_answer
0,Dense Llama3,Llama3,Dense,solar observatory aims to provide better under...,solar observatory aims to provide better under...,"According to the provided context, the Solar D..."
1,Dense Llama3,Llama3,Dense,mary j bliges new album my life ii,mary j bliges new album my life ii is a follow...,"According to the provided context, Mary J. Bli..."
2,Dense Llama3,Llama3,Dense,germany coach joachim low claims spain are fav...,germany coach joachim low claims spain are fav...,"Yes, according to the context, Germany coach J..."
3,Dense Llama3,Llama3,Dense,presidential historians weigh in on how histor...,presidential historians weigh in on how histor...,Presidential historians weigh in on how histor...
4,Dense Llama3,Llama3,Dense,chinas gold medal gymnasts cleared of competin...,chinas gold medal gymnasts cleared of competin...,"According to the provided context, China's gol..."


In [20]:
# ============================================================
# DATAFRAME INFORMATION
# ============================================================

results_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 53 entries, 0 to 52
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   pipeline           53 non-null     str  
 1   model              53 non-null     str  
 2   retrieval_method   53 non-null     str  
 3   query              53 non-null     str  
 4   reference_summary  53 non-null     str  
 5   generated_answer   53 non-null     str  
dtypes: str(6)
memory usage: 35.4 KB


In [21]:
# ============================================================
# CHECK NULL VALUES
# ============================================================

results_df.isnull().sum()

pipeline             0
model                0
retrieval_method     0
query                0
reference_summary    0
generated_answer     0
dtype: int64

In [22]:
# ============================================================
# MANUAL VALIDATION
# ============================================================

results_df[

    [

        "query",

        "reference_summary",

        "generated_answer"

    ]

]

,query,reference_summary,generated_answer
0,solar observatory aims to provide better under...,solar observatory aims to provide better under...,"According to the provided context, the Solar D..."
1,mary j bliges new album my life ii,mary j bliges new album my life ii is a follow...,"According to the provided context, Mary J. Bli..."
2,germany coach joachim low claims spain are fav...,germany coach joachim low claims spain are fav...,"Yes, according to the context, Germany coach J..."
3,presidential historians weigh in on how histor...,presidential historians weigh in on how histor...,Presidential historians weigh in on how histor...
4,chinas gold medal gymnasts cleared of competin...,chinas gold medal gymnasts cleared of competin...,"According to the provided context, China's gol..."
5,critics and viewers see stewart as victor after,critics and viewers see stewart as victor afte...,"According to the provided context, critics and..."
6,bahr idriss abu garda faces charges in deaths,bahr idriss abu garda faces charges in deaths ...,"Bahr Idriss Abu Garda faces charges of murder,..."
7,south africa beat cohosts india in world cup,south africa beat cohosts india in world cup c...,"Yes, according to the provided context, South ..."
8,rudy ruiz its become unfashionable to have an,rudy ruiz its become unfashionable to have an ...,It seems like you're asking me to complete Rud...
9,fabio cannavaro is to join the italian national,fabio cannavaro is to join the italian nationa...,the Italian national squad on Sunday for their...


In [23]:
# ============================================================
# SAMPLE OUTPUTS
# ============================================================

for i in range(5):

    print(
        "\n" + "=" * 80
    )

    print(
        f"Query {i+1}:"
    )

    print(
        results_df.iloc[i]["query"]
    )

    print(
        "\nReference Summary:\n"
    )

    print(
        results_df.iloc[i]["reference_summary"]
    )

    print(
        "\nGenerated Answer:\n"
    )

    print(
        results_df.iloc[i]["generated_answer"]
    )


Query 1:
solar observatory aims to provide better understanding of

Reference Summary:

solar observatory aims to provide better understanding of suns role in space weather observatory to deliver images with resolution 10 times better than highdef tv fiveyear mission will examine suns magnetic field and violent solar events nasa observatory will snap image of sun in eight wavelengths every 10 seconds

Generated Answer:

According to the provided context, the Solar Dynamics Observatory (SDO) aims to provide a better understanding of the dynamics of the sun and its role in space weather events such as solar flares.

Query 2:
mary j bliges new album my life ii

Reference Summary:

mary j bliges new album my life ii is a followup to an album from 17 years ago the singer says that recently she has learned to say you know what i am deserving you either know how to get around it or you just have to go through it blige says

Generated Answer:

According to the provided context, Mary J. Blige'

In [24]:
# ============================================================
# SAVE RESULTS
# ============================================================

results_df.to_csv(

    "dense_llama3_eval.csv",

    index=False

)

print(
    "Dense Llama3 Evaluation Results Saved Successfully"
)

Dense Llama3 Evaluation Results Saved Successfully


In [25]:
# ============================================================
# VERIFY SAVED FILE
# ============================================================

saved_df = pd.read_csv(
    "dense_llama3_eval.csv"
)

print(
    saved_df.shape
)

saved_df.head()

(53, 6)


,pipeline,model,retrieval_method,query,reference_summary,generated_answer
0,Dense Llama3,Llama3,Dense,solar observatory aims to provide better under...,solar observatory aims to provide better under...,"According to the provided context, the Solar D..."
1,Dense Llama3,Llama3,Dense,mary j bliges new album my life ii,mary j bliges new album my life ii is a follow...,"According to the provided context, Mary J. Bli..."
2,Dense Llama3,Llama3,Dense,germany coach joachim low claims spain are fav...,germany coach joachim low claims spain are fav...,"Yes, according to the context, Germany coach J..."
3,Dense Llama3,Llama3,Dense,presidential historians weigh in on how histor...,presidential historians weigh in on how histor...,Presidential historians weigh in on how histor...
4,Dense Llama3,Llama3,Dense,chinas gold medal gymnasts cleared of competin...,chinas gold medal gymnasts cleared of competin...,"According to the provided context, China's gol..."


In [26]:
# ============================================================
# FINAL SUMMARY
# ============================================================

print(
    "Pipeline : Dense Llama3"
)

print(
    "Rows Evaluated :",
    len(saved_df)
)

print(
    "Output File : dense_llama3_eval.csv"
)

print(
    "Evaluation Completed Successfully"
)

Pipeline : Dense Llama3
Rows Evaluated : 53
Output File : dense_llama3_eval.csv
Evaluation Completed Successfully
